# Unified EEG Analysis

This notebook runs both **ISC** (Inter-Subject Correlation) and **mean/variance**
analyses using the shared workflow from `scripts/analysis_common.py`.

Select which analyses to run by toggling the flags in the *Configuration* section.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from scripts.analysis_common import (
    load_analyzers,
    analyzers_to_datasets,
    run_isc_workflow,
    run_mean_variance_workflow,
    BAND_ISC_THRESHOLDS,
)
from src.definitions.fields import (
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
)
from src.definitions.constants import ProjectPaths

from xvfbwrapper import Xvfb

vdisplay = Xvfb()
vdisplay.start()

%matplotlib inline

## Configuration

Toggle `RUN_ISC` / `RUN_MEAN_VARIANCE` to select which analyses to run.

In [ ]:
# Which analyses to run
RUN_ISC = True
RUN_MEAN_VARIANCE = True

# Condition and music types
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC]

# ISC-specific settings
ISC_THRESHOLD = 0.035

# Sliding-window parameters (shared)
WINDOW_SEC = 5.0
STEP_SEC = 2.5

# Set to True to load raw files, resample, stack and save before analysis
process_and_save_data = False

## Data Loading

Load (or process & save) the analysers and create ``AnalysisData`` containers.
This step is shared across all analyses.

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES, CONDITION, EXCLUSION_CATEGORIES, process_and_save_data
)
datasets = analyzers_to_datasets(analyzers)

## ISC Analysis

Broadband LOO-ISC, sliding-window ISC, per-band ISC, and band-overlap analysis.

In [ ]:
if RUN_ISC:
    run_isc_workflow(
        datasets,
        save_dir=ProjectPaths.PLOTS_PATH / "OverallAnalysis",
        isc_threshold=ISC_THRESHOLD,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC,
    )

## Mean & Variance Analysis

Global and sliding-window mean/variance, plus per-band variants.

In [ ]:
if RUN_MEAN_VARIANCE:
    run_mean_variance_workflow(
        datasets,
        analyzers,
        save_dir=ProjectPaths.PLOTS_PATH / "MeanVarianceAnalysis",
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC,
    )